# 04. Hypothesis Experiments

H1/H2/H3 가설 검증 실험:
- **실험 1**: Performance–Dimension Scaling Curve (H1)
- **실험 2**: Principal Component 구조 분석 (H2a)
- **실험 3**: Linearity Test — Δ = Acc(RBF) - Acc(Linear) (H2b)
- **실험 4**: Equivalence Test — TOST (H3)

## 0. 데이터 로딩 (독립 실행용)

In [ ]:
# 01번 노트북을 먼저 실행하세요, 또는:
# %run ./01_Setup_and_Data_Loading.ipynb

## 설정

In [ ]:
from scipy.optimize import curve_fit
from scipy.stats import wilcoxon, ttest_1samp

ALL_5 = ['Ord_PI', 'Inter_PI', '3D_PI', 'Sixpack_Rips', 'Sixpack_Chroma']
SEEDS_H = [42, 123, 456, 789, 1010]
COLORS = {'Ord_PI':'#4C72B0','Inter_PI':'#DD8452','3D_PI':'#55A868',
          'Sixpack_Rips':'#C44E52','Sixpack_Chroma':'#8172B3'}

def eval_at_dim(X, y, pca_dim, clf, seed=42):
    X_s = StandardScaler().fit_transform(X)
    if pca_dim is not None and X_s.shape[1] > pca_dim:
        X_s = PCA(n_components=pca_dim, random_state=seed).fit_transform(X_s)
    skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=seed)
    accs = []
    for tri, tei in skf.split(X_s, y):
        c = clone(clf); c.fit(X_s[tri], y[tri])
        accs.append(accuracy_score(y[tei], c.predict(X_s[tei])))
    return np.mean(accs) * 100

## 실험 1: Performance–Dimension Scaling Curve (H1)

In [ ]:
print('=' * 80)
print('실험 1: Performance–Dimension Scaling Curve (H1)')
print('=' * 80)

DIMS_SWEEP = [16, 32, 64, 128, 256, 512, 1024]
svml = SVC(kernel='linear', C=1.0)
best_clfs = {'Ord_PI': RandomForestClassifier(100, random_state=42),
             'Inter_PI': RandomForestClassifier(100, random_state=42),
             '3D_PI': RandomForestClassifier(100, random_state=42),
             'Sixpack_Rips': SVC(kernel='rbf', C=1.0, gamma='scale'),
             'Sixpack_Chroma': SVC(kernel='rbf', C=1.0, gamma='scale')}

exp1 = {m: {'svml': {}, 'best': {}} for m in ALL_5}
for method in ALL_5:
    if method not in datasets: continue
    X, y = datasets[method]['X'], datasets[method]['y']
    print(f'\n[{method}] (원본 dim={X.shape[1]})')
    for dim in DIMS_SWEEP:
        if dim > X.shape[1]: continue
        svml_scores = [eval_at_dim(X, y, dim, svml, s) for s in SEEDS_H]
        best_scores = [eval_at_dim(X, y, dim, best_clfs[method], s) for s in SEEDS_H]
        exp1[method]['svml'][dim] = svml_scores
        exp1[method]['best'][dim] = best_scores
        print(f'  D={dim:<5d} SVM-L={np.mean(svml_scores):.2f}±{np.std(svml_scores):.2f}  '
              f'Best={np.mean(best_scores):.2f}±{np.std(best_scores):.2f}')

In [ ]:
# Saturation Fit & Figure
def sat_func(D, a, b, c): return a - b * np.power(D, -c)

print('\n--- Saturation Fit ---')
for method in ALL_5:
    dims = sorted(exp1[method]['svml'].keys())
    if not dims: continue
    means = [np.mean(exp1[method]['svml'][d]) for d in dims]
    try:
        popt, _ = curve_fit(sat_func, dims, means, p0=[80, 100, 0.5], maxfev=5000)
        print(f'  {method:<18s} a(asymptote)={popt[0]:.2f}%, c(rate)={popt[2]:.4f}')
    except: print(f'  {method:<18s} fit failed')

fig, axes = plt.subplots(1, 2, figsize=(16, 6))
for ax, mode, title in [(axes[0],'svml','(A) SVM-Linear'), (axes[1],'best','(B) Best Classifier')]:
    for m in ALL_5:
        dims = sorted(exp1[m][mode].keys())
        if not dims: continue
        means = [np.mean(exp1[m][mode][d]) for d in dims]
        stds = [np.std(exp1[m][mode][d]) for d in dims]
        ax.errorbar(dims, means, yerr=stds, fmt='o-', label=m,
                    color=COLORS[m], linewidth=2, markersize=6, capsize=3)
    ax.set_xlabel('Embedding Dimension D'); ax.set_ylabel('Strict Accuracy (%)')
    ax.set_title(f'Exp 1 {title}: Acc vs Dim', fontweight='bold')
    ax.legend(fontsize=9); ax.set_xscale('log', base=2); ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'exp1_perf_dim_curve.png'), dpi=150, bbox_inches='tight')
plt.show()

## 실험 2: Principal Component 구조 분석 (H2a)

In [ ]:
print('실험 2: Principal Component 구조 분석 (H2a)')
fig, axes = plt.subplots(1, 3, figsize=(20, 5))
d_effs = {}

# 2-1: Eigenvalue spectrum
ax = axes[0]
for m in ALL_5:
    if m not in datasets: continue
    X = StandardScaler().fit_transform(datasets[m]['X'])
    pca = PCA(n_components=min(500, X.shape[1]), random_state=42).fit(X)
    evals = pca.explained_variance_[:min(300, len(pca.explained_variance_))]
    ax.plot(range(1, len(evals)+1), evals/evals[0], label=m, color=COLORS[m], linewidth=1.5)
    lam = pca.explained_variance_
    d_effs[m] = (np.sum(lam)**2) / np.sum(lam**2)
ax.set_xlabel('PC Index'); ax.set_ylabel('Normalized Eigenvalue')
ax.set_title('Eigenvalue Spectrum', fontweight='bold')
ax.set_yscale('log'); ax.set_xscale('log'); ax.legend(fontsize=8); ax.grid(True, alpha=0.3)

# 2-2: Participation Ratio
ax = axes[1]; names = list(d_effs.keys()); vals = [d_effs[n] for n in names]
ax.bar(range(len(names)), vals, color=[COLORS[n] for n in names])
ax.set_xticks(range(len(names))); ax.set_xticklabels(names, rotation=30, ha='right', fontsize=9)
ax.set_ylabel('d_eff'); ax.set_title('Effective Dimensionality', fontweight='bold')
for i, v in enumerate(vals): ax.text(i, v+0.5, f'{v:.1f}', ha='center', fontsize=9, fontweight='bold')
ax.grid(True, alpha=0.3, axis='y')

# 2-3: Top-k PC Accuracy
ax = axes[2]; TOP_K = [5, 10, 20, 50, 100, 200, 500]
for m in ALL_5:
    if m not in datasets: continue
    X = StandardScaler().fit_transform(datasets[m]['X']); y = datasets[m]['y']
    X_pca = PCA(n_components=min(500, X.shape[1]), random_state=42).fit_transform(X)
    accs, ks_valid = [], []
    for k in TOP_K:
        if k > X_pca.shape[1]: continue
        ks_valid.append(k); X_k = X_pca[:, :k]
        skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
        fold_acc = [accuracy_score(y[tei], SVC(kernel='linear',C=1.).fit(X_k[tri],y[tri]).predict(X_k[tei]))
                    for tri, tei in skf.split(X_k, y)]
        accs.append(np.mean(fold_acc)*100)
    ax.plot(ks_valid, accs, 'o-', label=m, color=COLORS[m], linewidth=2, markersize=6)
ax.set_xlabel('Top-k PCs'); ax.set_ylabel('Strict Accuracy (%)')
ax.set_title('Top-k PC Accuracy (SVM-L)', fontweight='bold')
ax.set_xscale('log', base=2); ax.legend(fontsize=8); ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'exp2_pc_structure.png'), dpi=150, bbox_inches='tight')
plt.show()

# 요약
print(f"{'Method':<18s} {'d_eff':<10s} {'90%@PC':<10s} {'95%@PC':<10s} {'99%@PC':<10s}")
for m in ALL_5:
    if m not in datasets: continue
    X = StandardScaler().fit_transform(datasets[m]['X'])
    pca = PCA(n_components=min(500, X.shape[1]), random_state=42).fit(X)
    cumvar = np.cumsum(pca.explained_variance_ratio_)*100
    print(f'{m:<18s} {d_effs[m]:<10.1f} {np.searchsorted(cumvar,90)+1:<10d} '
          f'{np.searchsorted(cumvar,95)+1:<10d} {np.searchsorted(cumvar,99)+1:<10d}')

## 실험 3: Linearity Test — Δ = Acc(RBF) - Acc(Linear) (H2b)

In [ ]:
print('실험 3: Linearity Test (H2b)')
DIMS_LIN = [20, 50, 100, 200, 500]
exp3 = {m: [] for m in ALL_5}
for method in ALL_5:
    if method not in datasets: continue
    X, y = datasets[method]['X'], datasets[method]['y']
    print(f'\n[{method}]')
    for dim in DIMS_LIN:
        if dim > X.shape[1]: continue
        X_s = StandardScaler().fit_transform(X)
        X_pca = PCA(n_components=dim, random_state=42).fit_transform(X_s)
        skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
        acc_l, acc_rbf = [], []
        for tri, tei in skf.split(X_pca, y):
            cl = SVC(kernel='linear',C=1.); cl.fit(X_pca[tri],y[tri])
            acc_l.append(accuracy_score(y[tei], cl.predict(X_pca[tei])))
            cr = SVC(kernel='rbf',C=1.,gamma='scale'); cr.fit(X_pca[tri],y[tri])
            acc_rbf.append(accuracy_score(y[tei], cr.predict(X_pca[tei])))
        ml, mr = np.mean(acc_l)*100, np.mean(acc_rbf)*100
        exp3[method].append({'dim':dim,'svml':ml,'rbf':mr,'delta':mr-ml})
        print(f'  D={dim:<6d} SVM-L={ml:<10.2f} RBF={mr:<10.2f} Δ={mr-ml:+.2f}')

fig, ax = plt.subplots(figsize=(10, 5))
for m in ALL_5:
    if not exp3[m]: continue
    dims = [e['dim'] for e in exp3[m]]; deltas = [e['delta'] for e in exp3[m]]
    ax.plot(dims, deltas, 'o-', label=m, color=COLORS[m], linewidth=2, markersize=6)
ax.set_xlabel('PCA Dimension D'); ax.set_ylabel('Δ = Acc(RBF) - Acc(Linear) (%)')
ax.set_title('Linearity Gap Δ(D)', fontweight='bold')
ax.axhline(y=0, color='gray', linestyle='--', alpha=0.5)
ax.legend(); ax.set_xscale('log', base=2); ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'exp3_linearity_test.png'), dpi=150, bbox_inches='tight')
plt.show()

## 실험 4: Equivalence Test — TOST (H3)

In [ ]:
print('실험 4: Equivalence Test — TOST (H3)')

def tost_equivalence(scores_a, scores_b, delta=1.0):
    diffs = np.array(scores_a) - np.array(scores_b)
    t1, p1 = ttest_1samp(diffs + delta, 0, alternative='greater')
    t2, p2 = ttest_1samp(diffs - delta, 0, alternative='less')
    return np.mean(diffs), max(p1, p2)

DELTA = 2.0; N_REPEATS = 20
print(f'Equivalence margin δ = {DELTA}%, Repeats = {N_REPEATS}')

for D_high in [256, 512, None]:
    label = f'PCA={D_high}' if D_high else 'NoPCA'
    rips_scores, chroma_scores = [], []
    X_r, y_r = datasets['Sixpack_Rips']['X'], datasets['Sixpack_Rips']['y']
    X_c, y_c = datasets['Sixpack_Chroma']['X'], datasets['Sixpack_Chroma']['y']
    for seed in range(N_REPEATS):
        rips_scores.append(eval_at_dim(X_r, y_r, D_high, SVC(kernel='linear',C=1.), seed=seed))
        chroma_scores.append(eval_at_dim(X_c, y_c, D_high, SVC(kernel='linear',C=1.), seed=seed))
    mean_diff, p_tost = tost_equivalence(chroma_scores, rips_scores, delta=DELTA)
    equiv = '✓ EQUIVALENT' if p_tost < 0.05 else '✗ NOT equivalent'
    print(f'\n[{label}]  Rips: {np.mean(rips_scores):.2f}±{np.std(rips_scores):.2f}%  '
          f'Chroma: {np.mean(chroma_scores):.2f}±{np.std(chroma_scores):.2f}%  '
          f'Diff: {mean_diff:+.2f}%  TOST p={p_tost:.4f} → {equiv}')

## 전체 요약

In [ ]:
print('\n' + '=' * 60)
print('전체 요약')
print('=' * 60)
print(f"{'Method':<18s} {'d_eff':<8s} {'Δ@max':<10s}")
print('-' * 36)
for m in ALL_5:
    if m not in d_effs: continue
    delta_vals = [e['delta'] for e in exp3.get(m,[])]
    delta_str = f'{delta_vals[-1]:+.2f}' if delta_vals else 'N/A'
    print(f'{m:<18s} {d_effs[m]:<8.1f} {delta_str:<10s}')